# VarQITE — Portfolio Optimization (QAOA X-Mixer Ansatz)

Variational Quantum Imaginary Time Evolution via McLachlan's variational principle ([arXiv:1804.03023](https://arxiv.org/pdf/1804.03023))

Fundemental derivation of the McLachlan's variational principle and further Fubini-Studey phase correction (eq.13-14 [arXiv:1812.08767](https://arxiv.org/pdf/1812.08767))


$\frac{d|\psi(\tau)\rangle}{d\tau} = -(H - E_\tau)|\psi(\tau)\rangle$

$\sum_j A_{ij}\dot{\theta}_j = C_i \quad\Rightarrow\quad \theta \leftarrow \theta + \delta\tau\, A^{-1}C$

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import cudaq
from cudaq import spin
import sys
import os
import time
import torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from math import sqrt
from tqdm import tqdm
from typing import List, Tuple
sys.path.append(os.path.abspath(".."))
from Utils.qaoaCUDAQ import po_normalize, ret_cov_to_QUBO, qubo_to_ising, process_ansatz_values, kernel_qaoa_X, all_state_to_return, find_budget, to_sig

pd.set_option("display.width", 1000)

# Hyperparameters

INIT $\in$ {Zero, Random, Ramp}

OPTIMIZE_METHOD $\in$ {gradient, mcLachlan}

INVERSE_METHOD $\in$ {diagonalize, tikhonov} (mcLachlan only)

GRAD_METHOD $\in$ {fd, psr}: how $C$ is obtained (fd: finite difference of $\langle H\rangle$, psr: parameter-shift)

A_METHOD $\in$ {dpsi, fidelity, fidelity_kappa, diag}: how $A$ is obtained
- dpsi: FD of the statevector (reference only, not measurable)
- fidelity: overlap circuits, uniform shift FID_SHIFT, diagonal from the stencil (M1 with uniform $\delta$)
- fidelity_kappa: overlap circuits with per-parameter shifts $\delta_k = \min(\kappa/\sqrt{A_{kk}},\ \tfrac12\,\text{CAP\_ANGLE}/g_k)$, diagonal $A_{kk} = \mathrm{Var}_{\psi_k}(G_k)$ from truncated circuits (M1)
- diag: $\mathrm{Var}_{\psi_k}(G_k)$ only (M4, natural-gradient approximation)

FD_SCHEME $\in$ {forward, central}: stencil for $C$; FID_STENCIL $\in$ {forward, central}: stencil for the fidelity Hessian (central = 4-point, $O(\delta^2)$)

FID_SHIFT (uniform shift), KAPPA and CAP_ANGLE (per-parameter shifts, CAP_ANGLE bounds the rotation-angle shift in rad)

SHOTS: None for exact expectation values, or an int $S$ to emulate shot noise ($F \sim \mathrm{Bin}(S,F)/S$, variances from $S$ sampled bitstrings, energies $+\mathcal N(0, V_\tau/S)$)

The route-sweep driver `varqite_routes.py` mirrors this cell in its `DEFAULTS` (plus `EXACT_MEASURE`, `TARGET`) — keep the two in sync.


In [ ]:
e = 0
SEED = 0
N_ASSETS = 3
TARGET_QUBIT_IN = 2
q = 1.5
lamb = 0.005
LAYER = 5
eps = 0.1

INIT = "Ramp"
DELTA_GAMMA = 3.0
DELTA_BETA = 1.5

OPTIMIZE_METHOD = "mcLachlan"

DTAU = 0.1
N_STEPS = 300
FD_SHIFT = 1e-4
GRAD_METHOD = "fd"
PHASE_CORRECTION = True
A_METHOD = "fidelity_kappa"
FD_SCHEME = "forward"
FID_STENCIL = "central"
FID_SHIFT = 1e-2
KAPPA = 0.01
CAP_ANGLE = 0.02
SHOTS = None
# INVERSE_METHOD = "diagonalize"
INVERSE_METHOD = "tikhonov"
TIKHONOV_LAMBDA = 1e-7
EIG_CUTOFF = 1e-12

MAX_ITER = 300
LR = 0.01
SHIFT = 1e-4
WEIGHT_DECAY = 0.0

F_TOL = 1e-4
HAM_BOOST_MODE = "Jh"
HAM_BOOST = 1.0
PRECISION = "fp64"

min_P, max_P = 108, 216
DUPLICATE_ASSET = False

assert INIT in ["Zero", "Random", "Ramp"]
assert OPTIMIZE_METHOD in ["gradient", "mcLachlan"]
assert INVERSE_METHOD in ["diagonalize", "tikhonov"]
assert GRAD_METHOD in ["fd", "psr"]
assert A_METHOD in ["dpsi", "fidelity", "fidelity_kappa", "diag"]
assert FD_SCHEME in ["forward", "central"]
assert FID_STENCIL in ["forward", "central"]
assert HAM_BOOST_MODE in ["J", "Jh", "h", "fixed"]

if PRECISION == "fp64":
    cudaq.set_target("nvidia", option="fp64")
else:
    cudaq.set_target("nvidia")
device = torch.device("cuda:0")

# Dataset

In [ ]:
data_cov_pd = pd.read_csv("../dataset/top_50_us_stocks_data_20250526_011226_covariance.csv")
data_ret_p_pd = pd.read_csv("../dataset/top_50_us_stocks_returns_price.csv")

data_ret_p_pd = data_ret_p_pd[(data_ret_p_pd["Price"] > min_P) & (data_ret_p_pd["Price"] < max_P)]
data_cov_pd = data_cov_pd.loc[data_cov_pd["Ticker"].isin(data_ret_p_pd["Ticker"])].reset_index(drop=True)
data_cov_pd = data_cov_pd[["Ticker"] + data_cov_pd["Ticker"].tolist()]

In [ ]:
st = time.perf_counter()
np.random.seed(911 + 991 * e + 997 * N_ASSETS)
rng_state = np.random.get_state()
asset_idx = np.random.choice(data_cov_pd.shape[0], N_ASSETS, replace=DUPLICATE_ASSET)
data_cov = data_cov_pd.drop("Ticker", axis=1).to_numpy()[asset_idx, :][:, asset_idx]
stock_names = data_ret_p_pd["Company_Name"].to_numpy()[asset_idx]
data_ret_p = data_ret_p_pd.drop("Ticker", axis=1)
asset_idx_raw = data_ret_p.index[asset_idx].to_numpy()
data_ret_p = data_ret_p.drop("Company_Name", axis=1).to_numpy()[asset_idx, :]
data_ret = data_ret_p[:, 0]
data_p = data_ret_p[:, 1]

np.random.set_state(rng_state)
weighted = np.random.uniform(0, 1)
B_mi, B_ma = find_budget(TARGET_QUBIT_IN * N_ASSETS, data_p, min_P, max_P, min_mix_mode=True)
B = B_mi * weighted + B_ma * (1 - weighted)
data_time = time.perf_counter() - st

print("Selected Stocks:", stock_names)
print("Prices:", data_p)
print("Returns:", data_ret)
print("Budget:", B)

# Hamiltonians

$
\begin{aligned}
H &= \lambda I - \sum_{i,j} \text{QU}_{ij} \left(\frac{1-Z_i}{2}\right)\otimes\left(\frac{1-Z_j}{2}\right) \\
&= nI + \sum_i h_i Z_i + \sum_{ij} J_{ij} Z_i Z_j
\end{aligned}
$

Note that $QU$ is a max problem, while $H$ is now a min problem.

One way to normalize the Hamiltonian is:
$
H_\text{norm} = \frac{H - \lambda I}{\|H - \lambda I\|_F}
$

In [ ]:
st = time.perf_counter()
P = data_p[:N_ASSETS]
ret = data_ret[:N_ASSETS]
cov = data_cov[:N_ASSETS, :N_ASSETS]
P_bb, ret_bb, cov_bb, n_qubit, n_max, C_enc = po_normalize(B, P, ret, cov)
TARGET_QUBIT = n_qubit

QU = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, lamb, q)
QU_lamb = ret_cov_to_QUBO(np.zeros_like(ret_bb), np.zeros_like(cov_bb), P_bb, lamb, 0.0)
QU_eval = ret_cov_to_QUBO(ret_bb, cov_bb, P_bb, 0.0, q)
QU_return = ret_cov_to_QUBO(ret_bb, np.zeros_like(cov_bb), np.zeros_like(P_bb), 0.0, 0.0)
QU_risk = ret_cov_to_QUBO(np.zeros_like(ret_bb), cov_bb, np.zeros_like(P_bb), 0.0, q)

H_ansatz = -qubo_to_ising(QU, lamb).canonicalize()
idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use = process_ansatz_values(H_ansatz)
coeff_1_use, coeff_2_use = np.array(coeff_1_use), np.array(coeff_2_use)

max_J = np.max(np.abs(coeff_2_use))
max_h = np.max(np.abs(coeff_1_use))
max_J_h = max(max_J, max_h)
use_norm = max_J if HAM_BOOST_MODE == "J" else max_J_h if HAM_BOOST_MODE == "Jh" else max_h if HAM_BOOST_MODE == "h" else 1.0
hamiltonian_boost = to_sig(1 / use_norm if HAM_BOOST_MODE != "fixed" else HAM_BOOST, 4)
H_ansatz = H_ansatz * hamiltonian_boost
H_eval = -qubo_to_ising(QU_eval, 0.0).canonicalize() * hamiltonian_boost
H_lamb = -qubo_to_ising(QU_lamb, lamb).canonicalize() * hamiltonian_boost
H_return = -qubo_to_ising(QU_return, 0.0).canonicalize() * hamiltonian_boost
H_risk = -qubo_to_ising(QU_risk, 0.0).canonicalize() * hamiltonian_boost
ham_time = time.perf_counter() - st

print("Qubits:", n_qubit)
print("hamiltonian_boost:", hamiltonian_boost)
print("terms: h =", len(coeff_1_use), ", J =", len(coeff_2_use))

# Exact Reference

$H$ is diagonal in the computational basis, so $\mathrm{diag}(H)$, the exact ground state, and $H|\psi\rangle$ come from the classical energy table (no `to_matrix` / `eigh` needed)

In [ ]:
st = time.perf_counter()
state_eval = all_state_to_return(n_qubit, 0.0, QU_eval)
state_optim = -all_state_to_return(n_qubit, lamb, QU)
state_penalty = -all_state_to_return(n_qubit, lamb, QU_lamb)
diag_H = hamiltonian_boost * state_optim.astype(np.float64)

order = np.argsort(state_optim)
E0, E1 = diag_H[order[0]], diag_H[order[1]]
idx_ground = np.where(np.isclose(state_optim, state_optim[order[0]]))[0]
idx_optimal = int(np.argsort(state_eval)[-1])

eps_t = lamb * eps ** 2
idx_feasible = np.where(np.abs(state_penalty) <= eps_t)[0]
mi_r, ma_r = (state_eval[idx_feasible].min(), state_eval[idx_feasible].max()) if len(idx_feasible) >= 2 else (np.nan, np.nan)
exact_time = time.perf_counter() - st

print("E0:", E0, "| E1:", E1, "| spectral gap:", E1 - E0)
print("ground state(s):", [bin(i)[2:].zfill(n_qubit) for i in idx_ground])
print("optimal (eval) state:", bin(idx_optimal)[2:].zfill(n_qubit), "| is ground:", idx_optimal in set(idx_ground.tolist()))
print("feasible states:", len(idx_feasible), "| eval range:", (mi_r, ma_r))

# Ansatz & Initial Parameters

In [ ]:
layer_count = LAYER
parameter_count = 2 * layer_count
ansatz_fixed_param = (int(n_qubit), layer_count, idx_1_use, coeff_1_use, idx_2_a_use, idx_2_b_use, coeff_2_use)

mm_1 = np.min(np.abs(coeff_1_use)) if len(coeff_1_use) > 0 else 1e9
mm_2 = np.min(np.abs(coeff_2_use)) if len(coeff_2_use) > 0 else 1e9
mm_i = np.pi / min(mm_1, mm_2)

np.random.seed(4001 + 4099 * e + 4999 * N_ASSETS + 5099 * SEED)
points_init = np.zeros(parameter_count)
if INIT == "Random":
    points_init = np.random.uniform(-1, 1, parameter_count)
    points_init[:layer_count] *= mm_i
    points_init[layer_count:] *= np.pi
elif INIT == "Ramp":
    for i in range(layer_count):
        points_init[i] = DELTA_GAMMA * (i + 1) / layer_count
        points_init[layer_count + i] = DELTA_BETA * (1 - i / layer_count)
print("initial parameters:", np.round(points_init, 4).tolist())

In [ ]:
axes_flip = tuple(range(n_qubit - 1, -1, -1))
dim = 1 << n_qubit

def get_psi(params):
    psi = np.array(cudaq.get_state(kernel_qaoa_X, params, *ansatz_fixed_param))
    return psi.reshape([2] * n_qubit).transpose(axes_flip).ravel().astype(np.complex128)

def observe_energy(H, params):
    return float(cudaq.observe(kernel_qaoa_X, H, params, *ansatz_fixed_param).expectation())

def metrics(params, psi):
    prob = np.abs(psi) ** 2
    E_obj = observe_energy(H_ansatz, params) / hamiltonian_boost
    E_eval = observe_energy(H_eval, params) / hamiltonian_boost
    E_lamb = observe_energy(H_lamb, params) / hamiltonian_boost
    P_ground = float(prob[idx_ground].sum())
    P_optimal = float(prob[idx_optimal])
    approx_ratio = (float(prob @ state_eval) - mi_r) / (ma_r - mi_r) if len(idx_feasible) >= 2 else float("nan")
    return E_obj, E_eval, E_lamb, P_ground, P_optimal, approx_ratio

psi0 = get_psi(points_init)
print("energy via diag_H: ", float(np.real(np.vdot(psi0, diag_H * psi0))))
print("energy via observe:", observe_energy(H_ansatz, points_init))
print("E_lamb via diag:   ", float((np.abs(psi0) ** 2) @ state_penalty))
print("E_lamb via observe:", observe_energy(H_lamb, points_init) / hamiltonian_boost)

# McLachlan VarQITE

Note that
$
\partial_i\langle\phi|\phi\rangle = \langle\partial_i\phi|\phi\rangle + \langle\phi|\partial_i\phi\rangle = 0 \;
\Rightarrow\;
\langle\partial_i\phi|\phi\rangle = -\langle\phi|\partial_i\phi\rangle,
$

$
\begin{aligned}
A_{ij} &= \mathrm{Re}\big(\langle\partial_i\psi|\partial_j\psi\rangle\big) + \mathrm{Re}\big(\langle\partial_i\psi|\psi\rangle\langle\partial_j\psi|\psi\rangle\big) \\ &= \mathrm{Re}\big(\langle\partial_i\psi|\partial_j\psi\rangle\big) - \underbrace{\mathrm{Re}\big(\langle\partial_i\psi|\psi\rangle\langle\psi|\partial_j\psi\rangle\big)}_{\text{PHASE\_CORRECTION}}
\end{aligned}
$


$C_i = -\mathrm{Re}\big(\langle\partial_i\psi|H|\psi\rangle\big)$

diagonalize: $A = V\Lambda V^T$, drop $\lambda_k \le \text{EIG\_CUTOFF}\cdot\lambda_{\max}$, then $\dot\theta = V\Lambda^{+}V^T C$

tikhonov: $\dot\theta = (A + \lambda_{tik} I)^{-1} C$


# Analytic Differential

Let $|\psi(\beta,\gamma)\rangle = M(\beta_p)C(\gamma_p)\cdots M(\beta_1)C(\gamma_1)|+\rangle^{\otimes n}$ ,where

$
\begin{aligned}
C(\gamma) &= \exp\left(-i \gamma H_C\right) \\
M(\beta) &= \exp\left(-i \beta H_M\right)
\end{aligned}
$

# Measured $A$ and $C$ (no statevector access)

$C_i = -\tfrac12\,\partial_i\langle H\rangle$ exactly ($H$ Hermitian), so $C$ needs only energies:
forward $C_i \approx -\frac{E(\theta+\delta e_i)-E(\theta)}{2\delta}$ ($p+1$ observes), central $C_i \approx -\frac{E(\theta+\delta e_i)-E(\theta-\delta e_i)}{4\delta}$ ($2p$ observes).

$A$ is the Fubini-Study metric = Hessian of the fidelity $F(\theta') = |\langle\psi(\theta)|\psi(\theta')\rangle|^2$ at $\theta' = \theta$:

$
F(\theta + d\theta) = 1 - \sum_{ij} A_{ij}\, d\theta_i\, d\theta_j + O(d\theta^3)
\;\Rightarrow\; A = -\tfrac12\nabla^2 F
$

(gauge invariant, so the PHASE_CORRECTION term is automatically included). With $F_{i} = F(\theta+\delta e_i)$, $F_{ij} = F(\theta+\delta e_i+\delta e_j)$:

forward: $A_{ii} \approx \dfrac{1 - F_i}{\delta^2}$, $A_{ij} \approx \dfrac{F_i + F_j - F_{ij} - 1}{2\delta^2}$ ($p + p(p-1)/2$ overlaps, $O(\delta)$ error)

central: $A_{ii} \approx \dfrac{2 - F_{+i} - F_{-i}}{2\delta^2}$, $A_{ij} \approx -\dfrac{F_{+i+j} - F_{+i-j} - F_{-i+j} + F_{-i-j}}{8\delta^2}$ ($2p + 2p(p-1)$ overlaps, $O(\delta^2)$ error)

Note $2 - F_i - F_j \approx (A_{ii} + A_{jj})\,\delta^2$ carries no information about $A_{ij}$; the mixed shift $F_{ij}$ is required.

$F$ is measured with a compute-uncompute circuit: $F = |\langle 0|U(\theta)^\dagger U(\theta')|0\rangle|^2 = \langle P_0\rangle$, $P_0 = \prod_j \tfrac12(I + Z_j)$ (`kernel_qaoa_X_overlap` + `observe`).

Since $1 - F_i \approx A_{ii}\delta^2$ and $A \sim 10^{-5}$ here (unboosted circuit coefficients), FID_SHIFT must be $\gg$ FD_SHIFT to resolve $F$ in fp64.

**Diagonal from a variance.** With $|\psi_k\rangle$ the state just after the layer of $\theta_k$ and $G_k$ its generator ($G_\gamma = \sum_a c_a Z_a + \sum c_{ab} Z_aZ_b$ in circuit units, $G_\beta = \sum_a X_a$):
$A_{kk} = \mathrm{Var}_{\psi_k}(G_k)$, one truncated circuit per parameter (`kernel_qaoa_X_trunc`), measured in the $Z$ ($\gamma$) or $X$ ($\beta$) basis.
It also fixes the per-parameter shift for the stencil, $\delta_k = \kappa/\sqrt{A_{kk}}$ (relative bias $O(\kappa^2)$), capped so that no gate angle shifts by more than CAP_ANGLE.
This matters here because $A$ is extremely anisotropic: $A_{\gamma\gamma}\sim10^{-7}$ (coefficients $\sim 10^{-4}$) while $A_{\beta\beta}$ grows from $10^{-5}$ at the $|+\rangle$-like init to $O(1)$ along the flow.

**Consistency check.** $V_\tau = \langle H^2\rangle - E^2$ costs no extra circuit (same bitstrings as $E$). The minimum McLachlan residual
$\mathcal R_{\min} = V_\tau - C^{\mathsf T}A^{-1}C \ge 0$ and the cooling rate $dE/d\tau = -2\,C^{\mathsf T}A^{-1}C$ are logged per step (history columns 8, 9); $\mathcal R_{\min}<0$ flags an inconsistent (noisy / under-regularized) $A$.


In [ ]:
def dpsi_all(params):
    dps = np.zeros((parameter_count, dim), dtype=np.complex128)
    for i in range(parameter_count):
        pp, pm = params.copy(), params.copy()
        pp[i] += FD_SHIFT
        pm[i] -= FD_SHIFT
        dps[i] = (get_psi(pp) - get_psi(pm)) / (2 * FD_SHIFT)
    return dps

def build_A_dpsi(psi, dps):
    A = np.real(dps.conj() @ dps.T)
    if PHASE_CORRECTION:
        v = dps.conj() @ psi
        A -= np.real(np.outer(v, v.conj()))
    return A

rng = np.random.default_rng(SEED)

def psd(A):
    w, V = np.linalg.eigh(A)
    return V @ (np.abs(w) * V.T)

@cudaq.kernel
def kernel_qaoa_X_overlap(thetas_a: List[float], thetas_b: List[float], qubit_count: int, layer_count: int, idx_1: List[int], coeff_1: List[float], idx_2_a: List[int], idx_2_b: List[int], coeff_2: List[float]):
    # U(thetas_b) followed by U(thetas_a)^dagger: P(|0...0>) = |<psi(a)|psi(b)>|^2
    qreg = cudaq.qvector(qubit_count)
    h(qreg)
    for i in range(layer_count):
        for j in range(len(idx_1)):
            rz(2 * coeff_1[j] * thetas_b[i], qreg[idx_1[j]])
        for j in range(len(idx_2_a)):
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
            rz(2 * coeff_2[j] * thetas_b[i], qreg[idx_2_b[j]])
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
        for j in range(qubit_count):
            rx(2.0 * thetas_b[layer_count + i], qreg[j])
    for i in range(layer_count - 1, -1, -1):
        for j in range(qubit_count):
            rx(-2.0 * thetas_a[layer_count + i], qreg[j])
        for j in range(len(idx_2_a) - 1, -1, -1):
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
            rz(-2 * coeff_2[j] * thetas_a[i], qreg[idx_2_b[j]])
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
        for j in range(len(idx_1) - 1, -1, -1):
            rz(-2 * coeff_1[j] * thetas_a[i], qreg[idx_1[j]])
    h(qreg)

@cudaq.kernel
def kernel_qaoa_X_trunc(thetas: List[float], qubit_count: int, layer_count: int, idx_1: List[int], coeff_1: List[float], idx_2_a: List[int], idx_2_b: List[int], coeff_2: List[float], n_cost: int, n_mix: int, x_basis: int):
    # first n_cost cost layers and n_mix mixer layers (n_mix in {n_cost - 1, n_cost}); x_basis=1 rotates to the X basis before measuring
    qreg = cudaq.qvector(qubit_count)
    h(qreg)
    for i in range(n_cost):
        for j in range(len(idx_1)):
            rz(2 * coeff_1[j] * thetas[i], qreg[idx_1[j]])
        for j in range(len(idx_2_a)):
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
            rz(2 * coeff_2[j] * thetas[i], qreg[idx_2_b[j]])
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
        if i < n_mix:
            for j in range(qubit_count):
                rx(2.0 * thetas[layer_count + i], qreg[j])
    if x_basis == 1:
        h(qreg)

P_zero = 1.0
for j in range(n_qubit):
    P_zero = P_zero * (0.5 * (spin.i(j) + spin.z(j)))
G_gamma = H_ansatz * (1.0 / hamiltonian_boost)   # generator of gamma in circuit units: rz(2 c gamma) = exp(-i c gamma Z)
G_gamma2 = G_gamma * G_gamma
G_beta = spin.x(0)                                # generator of beta: rx(2 beta) = exp(-i beta X)
for j in range(1, n_qubit):
    G_beta = G_beta + spin.x(j)
G_beta2 = G_beta * G_beta
gen_scale = np.array([max_J_h] * layer_count + [1.0] * layer_count)   # largest gate angle per unit of parameter

def fidelity(thetas_a, thetas_b):
    F = float(cudaq.observe(kernel_qaoa_X_overlap, P_zero, thetas_a, thetas_b, *ansatz_fixed_param).expectation())
    return rng.binomial(SHOTS, min(max(F, 0.0), 1.0)) / SHOTS if SHOTS else F

def trunc_args(k):
    return (k + 1, k, G_gamma, G_gamma2) if k < layer_count else (k - layer_count + 1, k - layer_count + 1, G_beta, G_beta2)

def generator_of_bits(k, bitstr):                 # G_k evaluated on a measured bitstring (char j = qubit j)
    z = 1 - 2 * np.array([int(ch) for ch in bitstr])
    if k >= layer_count:
        return float(z.sum())
    return float(coeff_1_use @ z[idx_1_use] + coeff_2_use @ (z[idx_2_a_use] * z[idx_2_b_use]))

def var_diag(params):                             # A_kk = Var_{psi_k}(G_k): 2 observes (exact) or 1 sampled circuit (SHOTS) per parameter
    out = np.zeros(parameter_count)
    for k in range(parameter_count):
        n_cost, n_mix, G, G2 = trunc_args(k)
        if SHOTS:
            res = cudaq.sample(kernel_qaoa_X_trunc, params, *ansatz_fixed_param, n_cost, n_mix, 1 if k >= layer_count else 0, shots_count=int(SHOTS))
            vals = np.array([generator_of_bits(k, b) for b in res]); w = np.array([res.count(b) for b in res], dtype=float)
            m1 = w @ vals / w.sum()
            out[k] = w @ (vals - m1) ** 2 / w.sum()
        else:
            m1 = cudaq.observe(kernel_qaoa_X_trunc, G, params, *ansatz_fixed_param, n_cost, n_mix, 0).expectation()
            m2 = cudaq.observe(kernel_qaoa_X_trunc, G2, params, *ansatz_fixed_param, n_cost, n_mix, 0).expectation()
            out[k] = m2 - m1 ** 2
    return np.maximum(out, 0.0)

def kappa_delta(diag):
    return np.minimum(KAPPA / np.sqrt(np.maximum(diag, 1e-300)), 0.5 * CAP_ANGLE / gen_scale)

def build_A_fidelity(params, delta=None, diag=None):
    delta = np.full(parameter_count, FID_SHIFT) if delta is None else delta
    E = np.eye(parameter_count) * delta[:, None]
    F = lambda s: fidelity(params, params + s)
    A = np.zeros((parameter_count, parameter_count))
    if FID_STENCIL == "forward":
        F_i = np.array([F(E[i]) for i in range(parameter_count)])
        for i in range(parameter_count):
            A[i, i] = (1 - F_i[i]) / delta[i] ** 2
            for j in range(i + 1, parameter_count):
                A[i, j] = A[j, i] = (F_i[i] + F_i[j] - F(E[i] + E[j]) - 1) / (2 * delta[i] * delta[j])
    else:
        for i in range(parameter_count):
            if diag is None:
                A[i, i] = (2 - F(E[i]) - F(-E[i])) / (2 * delta[i] ** 2)
            for j in range(i + 1, parameter_count):
                A[i, j] = A[j, i] = -(F(E[i] + E[j]) - F(E[i] - E[j]) - F(-E[i] + E[j]) + F(-E[i] - E[j])) / (8 * delta[i] * delta[j])
    if diag is not None:
        A[np.diag_indices(parameter_count)] = diag
    return psd(A) if SHOTS else A

def n_circuits_A(method=None, stencil=None):    # circuits per step charged to A
    method = A_METHOD if method is None else method; stencil = FID_STENCIL if stencil is None else stencil
    p = parameter_count
    if method == "dpsi": return 2 * p
    if method == "diag": return 2 * p
    off = p + p * (p - 1) // 2 if stencil == "forward" else 2 * p * (p - 1)
    return off + (2 * p if method == "fidelity_kappa" else (0 if stencil == "forward" else 2 * p))

def build_A(params, psi=None):
    if A_METHOD == "fidelity":
        return build_A_fidelity(params)
    if A_METHOD == "fidelity_kappa":
        d = var_diag(params)
        return build_A_fidelity(params, kappa_delta(d), d)
    if A_METHOD == "diag":
        return np.diag(var_diag(params))
    return build_A_dpsi(get_psi(params) if psi is None else psi, dpsi_all(params))

def vtau(psi):
    prob = np.abs(psi) ** 2
    return float(prob @ diag_H ** 2 - (prob @ diag_H) ** 2)

@cudaq.kernel
def kernel_qaoa_X_shift(thetas: List[float], qubit_count: int, layer_count: int, idx_1: List[int], coeff_1: List[float], idx_2_a: List[int], idx_2_b: List[int], coeff_2: List[float], shift_layer: int, shift_gamma: int, shift_idx: int, shift_angle: float):
    qreg = cudaq.qvector(qubit_count)
    h(qreg)
    for i in range(layer_count):
        for j in range(len(idx_1)):
            rz(2 * coeff_1[j] * thetas[i], qreg[idx_1[j]])
        for j in range(len(idx_2_a)):
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
            rz(2 * coeff_2[j] * thetas[i], qreg[idx_2_b[j]])
            x.ctrl(qreg[idx_2_a[j]], qreg[idx_2_b[j]])
        if i == shift_layer:
            if shift_gamma == 1:
                if shift_idx < len(idx_1):
                    rz(shift_angle, qreg[idx_1[shift_idx]])
                else:
                    x.ctrl(qreg[idx_2_a[shift_idx - len(idx_1)]], qreg[idx_2_b[shift_idx - len(idx_1)]])
                    rz(shift_angle, qreg[idx_2_b[shift_idx - len(idx_1)]])
                    x.ctrl(qreg[idx_2_a[shift_idx - len(idx_1)]], qreg[idx_2_b[shift_idx - len(idx_1)]])
        for j in range(qubit_count):
            rx(2.0 * thetas[layer_count + i], qreg[j])
        if i == shift_layer:
            if shift_gamma == 0:
                rx(shift_angle, qreg[shift_idx])

def grad_psr(params, H):
    n_h = len(idx_1_use)
    n_terms = n_h + len(idx_2_a_use)
    grad = np.zeros(parameter_count)
    for l in range(layer_count):
        g = 0.0
        for k in range(n_terms):
            c_k = coeff_1_use[k] if k < n_h else coeff_2_use[k - n_h]
            f_p = cudaq.observe(kernel_qaoa_X_shift, H, params, *ansatz_fixed_param, l, 1, k, np.pi / 2).expectation()
            f_m = cudaq.observe(kernel_qaoa_X_shift, H, params, *ansatz_fixed_param, l, 1, k, -np.pi / 2).expectation()
            g += c_k * (f_p - f_m)
        grad[l] = g
        g = 0.0
        for j in range(n_qubit):
            f_p = cudaq.observe(kernel_qaoa_X_shift, H, params, *ansatz_fixed_param, l, 0, j, np.pi / 2).expectation()
            f_m = cudaq.observe(kernel_qaoa_X_shift, H, params, *ansatz_fixed_param, l, 0, j, -np.pi / 2).expectation()
            g += f_p - f_m
        grad[layer_count + l] = g
    return grad

def build_C(params):
    if GRAD_METHOD == "psr":
        return -0.5 * grad_psr(params, H_ansatz)
    sigma = np.sqrt(vtau(get_psi(params)) / SHOTS) if SHOTS else 0.0   # shot noise on each energy (noise model only)
    E = lambda p: observe_energy(H_ansatz, p) + (rng.normal(0.0, sigma) if SHOTS else 0.0)
    C = np.zeros(parameter_count)
    E_0 = E(params) if FD_SCHEME == "forward" else None
    for i in range(parameter_count):
        pp, pm = params.copy(), params.copy()
        pp[i] += FD_SHIFT
        pm[i] -= FD_SHIFT
        if FD_SCHEME == "forward":
            C[i] = -0.5 * (E(pp) - E_0) / FD_SHIFT
        else:
            C[i] = -0.5 * (E(pp) - E(pm)) / (2 * FD_SHIFT)
    return C

def solve_theta_dot(A, C):
    if INVERSE_METHOD == "diagonalize":
        w, V = np.linalg.eigh(A)
        w_inv = np.zeros_like(w)
        keep = w > EIG_CUTOFF * np.max(np.abs(w))
        w_inv[keep] = 1.0 / w[keep]
        return V @ (w_inv * (V.T @ C))
    return np.linalg.solve(A + TIKHONOV_LAMBDA * np.eye(parameter_count), C)

In [ ]:
psi0 = get_psi(points_init)
dps0 = dpsi_all(points_init)
A_dpsi = build_A_dpsi(psi0, dps0)
g_, b_ = slice(0, layer_count), slice(layer_count, parameter_count)
blk = lambda M, a, b: np.abs(M[a, b]).max()
sq = np.sqrt(np.outer(np.diag(A_dpsi), np.diag(A_dpsi)))
print(f"A_dpsi: |gamma-block| {blk(A_dpsi, g_, g_):.2e} | |beta-block| {blk(A_dpsi, b_, b_):.2e} | |cross| {blk(A_dpsi, g_, b_):.2e} | cond {np.linalg.cond(A_dpsi):.1e}")
print("A_dpsi eigenvalues:", np.array2string(np.linalg.eigvalsh(A_dpsi), precision=2))

thb = points_init + np.random.default_rng(0).uniform(-0.5, 0.5, parameter_count)
print(f"overlap circuit vs statevector |<psi_a|psi_b>|^2: {abs(fidelity(points_init, thb) - abs(np.vdot(psi0, get_psi(thb))) ** 2):.2e}")
d_var = var_diag(points_init)
print(f"Var(G_k) diagonal vs statevector diagonal: max rel err {np.abs(d_var - np.diag(A_dpsi)).max() / np.abs(np.diag(A_dpsi)).max():.1e} | kappa shifts gamma {kappa_delta(d_var)[:layer_count].max():.3g}, beta {kappa_delta(d_var)[layer_count:].max():.3g}")

_save = (A_METHOD, FID_STENCIL, FID_SHIFT)
print(f"{'A route':32s} | {'circuits':>8s} {'ms':>6s} | {'rel Frobenius':>13s} {'norm max |dA|/sqrt(A_ii A_jj)':>30s} | {'rel err gamma':>13s} {'beta':>8s} {'cross':>8s}")
for A_METHOD, FID_STENCIL, FID_SHIFT in [("fidelity", "forward", 1e-3), ("fidelity", "forward", 1e-2), ("fidelity", "central", 1e-3), ("fidelity", "central", 1e-2), ("fidelity_kappa", "forward", 1e-2), ("fidelity_kappa", "central", 1e-2), ("diag", "central", 1e-2)]:
    st = time.perf_counter()
    A_f = build_A(points_init)
    tt = (time.perf_counter() - st) * 1e3
    err = np.abs(A_f - A_dpsi)
    tag = A_METHOD + (f" {FID_STENCIL} shift {FID_SHIFT:.0e}" if A_METHOD == "fidelity" else f" {FID_STENCIL} kappa {KAPPA} cap {CAP_ANGLE}" if A_METHOD == "fidelity_kappa" else "")
    print(f"{tag:32s} | {n_circuits_A():8d} {tt:6.0f} | {np.linalg.norm(err) / np.linalg.norm(A_dpsi):13.1e} {(err / sq).max():30.1e} | {blk(err, g_, g_) / blk(A_dpsi, g_, g_):13.1e} {blk(err, b_, b_) / blk(A_dpsi, b_, b_):8.1e} {blk(err, g_, b_) / blk(A_dpsi, g_, b_):8.1e}")
A_METHOD, FID_STENCIL, FID_SHIFT = _save
st = time.perf_counter(); dpsi_all(points_init); print(f"{'dpsi (statevector reference)':32s} | {2 * parameter_count:8d} {(time.perf_counter() - st) * 1e3:6.0f}")

C_state = -np.real(dps0.conj() @ (diag_H * psi0))
_scheme = FD_SCHEME
for FD_SCHEME in ["forward", "central"]:
    st = time.perf_counter()
    C_o = build_C(points_init)
    print(f"C {FD_SCHEME:7s} ({parameter_count + 1 if FD_SCHEME == 'forward' else 2 * parameter_count:2d} observe, {(time.perf_counter() - st) * 1e3:.1f} ms): max |C_observe - C_state| = {np.abs(C_o - C_state).max():.2e}")
FD_SCHEME = _scheme
st = time.perf_counter()
C_psr = -0.5 * grad_psr(points_init, H_ansatz)
print(f"C psr     ({2 * layer_count * (len(coeff_1_use) + len(coeff_2_use) + n_qubit)} observe, {(time.perf_counter() - st) * 1e3:.1f} ms): max |C_psr - C_state|     = {np.abs(C_psr - C_state).max():.2e}")


In [ ]:
def run_mcLachlan(params_init, pbar=True):
    params = params_init.copy()
    history, iter_times = [], []
    last_f, cou_con, num_iter = None, 0, 0
    psi = get_psi(params)  # diagnostics only (P_ground etc.); feeds the update only when A_METHOD == "dpsi"
    pbar_it = tqdm(range(N_STEPS), disable=not pbar)
    for it in pbar_it:
        st_it = time.perf_counter()
        A = build_A(params, psi)
        C = build_C(params)
        theta_dot = solve_theta_dot(A, C)
        cooling = float(C @ theta_dot)                     # C^T A^+ C
        R_min = vtau(psi) - cooling                        # McLachlan minimum residual, must be >= 0
        params += DTAU * theta_dot
        psi = get_psi(params)
        iter_times.append(time.perf_counter() - st_it)
        E_obj, E_eval, E_lamb, P_ground, P_optimal, approx_ratio = metrics(params, psi)
        history.append([E_obj, E_eval, E_lamb, P_ground, P_optimal, approx_ratio, params[0], params[layer_count], R_min, -2 * cooling])
        num_iter += 1
        expectation = E_obj * hamiltonian_boost
        cou_con = cou_con + 1 if last_f is not None and abs(expectation - last_f) < F_TOL else 0
        if cou_con >= 3:
            break
        last_f = expectation
        if pbar:
            pbar_it.set_description(f"E {E_obj:.6f}, E_eval {E_eval:.6f}, P_gs {P_ground:.4f}")
    return params, np.array(history), np.array(iter_times), num_iter

# Gradient (old method)

Adam(lr=LR, betas=(0.95, 0.98)) + CosineAnnealingLR + forward-difference gradient of $\langle H\rangle$, as in PO_new_ApproxRatio.py

In [ ]:
def run_gradient(params_init, pbar=True):
    points_cu = torch.tensor(params_init, dtype=torch.float64, device=device)
    optimizer_cu = Adam([points_cu], lr=LR, betas=(0.95, 0.98), weight_decay=WEIGHT_DECAY, decoupled_weight_decay=True)
    scheduler = CosineAnnealingLR(optimizer_cu, T_max=MAX_ITER, eta_min=0.0003)
    history, iter_times = [], []
    last_f, cou_con, num_iter = None, 0, 0
    pbar_it = tqdm(range(MAX_ITER), disable=not pbar)
    for it in pbar_it:
        st_it = time.perf_counter()
        optimizer_cu.zero_grad()
        params = points_cu.detach().cpu().numpy()
        expectation = float(cudaq.observe(kernel_qaoa_X, H_ansatz, params, *ansatz_fixed_param).expectation())
        if GRAD_METHOD == "psr":
            grad = torch.tensor(grad_psr(params, H_ansatz), dtype=torch.float64, device=device)
        else:
            grad = torch.zeros(parameter_count, dtype=torch.float64, device=device)
            for j in range(parameter_count):
                shift = np.zeros(parameter_count)
                shift[j] = SHIFT
                forward = float(cudaq.observe(kernel_qaoa_X, H_ansatz, params + shift, *ansatz_fixed_param).expectation())
                grad[j] = (forward - expectation) / SHIFT
        points_cu.grad = grad
        optimizer_cu.step()
        scheduler.step()
        iter_times.append(time.perf_counter() - st_it)
        E_obj, E_eval, E_lamb, P_ground, P_optimal, approx_ratio = metrics(params, get_psi(params))
        history.append([E_obj, E_eval, E_lamb, P_ground, P_optimal, approx_ratio, points_cu[0].item(), points_cu[layer_count].item()])
        num_iter += 1
        cou_con = cou_con + 1 if last_f is not None and abs(expectation - last_f) < F_TOL else 0
        if cou_con >= 3:
            break
        last_f = expectation
        if pbar:
            pbar_it.set_description(f"E {E_obj:.6f}, E_eval {E_eval:.6f}, P_gs {P_ground:.4f}, LR {optimizer_cu.param_groups[0]['lr']:.4f}")
    return points_cu.detach().cpu().numpy(), np.array(history), np.array(iter_times), num_iter

# Optimize

In [ ]:
st = time.perf_counter()
if OPTIMIZE_METHOD == "mcLachlan":
    optimal_parameters, history, iter_times, num_iter = run_mcLachlan(points_init)
else:
    optimal_parameters, history, iter_times, num_iter = run_gradient(points_init)
optim_time = time.perf_counter() - st

print("method:", OPTIMIZE_METHOD + (f" ({INVERSE_METHOD})" if OPTIMIZE_METHOD == "mcLachlan" else ""))
print("iterations:", num_iter)
print(f"optim_time: {optim_time:.3f} s | per-iter: {iter_times.mean() * 1e3:.2f} ms (+- {iter_times.std() * 1e3:.2f})")
print("optimal parameters:", np.round(optimal_parameters, 4).tolist())
if history.shape[1] > 8:
    print(f"A: {A_METHOD} ({FID_STENCIL}) {n_circuits_A()} circuits/step | C: {GRAD_METHOD} ({FD_SCHEME}) | R_min < 0 in {int((history[:, 8] < 0).sum())}/{num_iter} steps | final dE/dtau {history[-1, 9]:.3e}")


# Results

In [ ]:
st = time.perf_counter()
psi_final = get_psi(optimal_parameters)
prob = np.abs(psi_final) ** 2
E_obj, E_eval, E_lamb, P_ground, P_optimal, approx_ratio = metrics(optimal_parameters, psi_final)
idx_best = int(np.argmax(prob))
maxprob_ratio = (state_eval[idx_best] - mi_r) / (ma_r - mi_r) if len(idx_feasible) >= 2 else float("nan")
budget_violation = sqrt(max(E_lamb, 0.0) / lamb)
return_final = -observe_energy(H_return, optimal_parameters) / hamiltonian_boost
risk_final = observe_energy(H_risk, optimal_parameters) / hamiltonian_boost
observe_time = time.perf_counter() - st

report = pd.DataFrame([{
    "Assets": N_ASSETS,
    "Qubits": n_qubit,
    "Layer": LAYER,
    "Exp": e,
    "Seed": SEED,
    "Method": OPTIMIZE_METHOD,
    "Inverse": INVERSE_METHOD if OPTIMIZE_METHOD == "mcLachlan" else "-",
    "Init": INIT,
    "Boost": hamiltonian_boost,
    "Energy": E_obj * hamiltonian_boost,
    "E0": E0,
    "Energy_Eval": E_eval,
    "Approximate_ratio": approx_ratio,
    "Prob_Ground": P_ground,
    "Prob_Optimal": P_optimal,
    "MaxProb_ratio": maxprob_ratio,
    "Return": return_final,
    "Risk": risk_final,
    "Budget_Violations": budget_violation,
    "Budget": B,
    "epochs": num_iter,
    "data_time": data_time,
    "ham_time": ham_time,
    "exact_time": exact_time,
    "optim_time": optim_time,
    "iter_time_ms": iter_times.mean() * 1e3,
    "observe_time": observe_time,
}])
print(report.T)
print()
print("most probable state:", bin(idx_best)[2:].zfill(n_qubit), "| prob:", round(float(prob[idx_best]), 4), "| feasible:", idx_best in set(idx_feasible.tolist()), "| ground:", idx_best in set(idx_ground.tolist()))

# Plots

In [ ]:
plt.figure(figsize=(18, 9))

plt.subplot(2, 3, 1)
plt.plot(history[:, 0] * hamiltonian_boost)
plt.axhline(y=E0, color="r", linestyle="--", label="E0")
plt.axhline(y=E1, color="orange", linestyle="--", label="E1")
plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.title("<H> (boosted)"); plt.legend()

plt.subplot(2, 3, 2)
plt.plot(history[:, 3], label="P_ground")
plt.plot(history[:, 4], label="P_optimal")
plt.axhline(y=1 / dim, color="r", linestyle="--", label="uniform")
plt.xlabel("Iteration"); plt.ylabel("Probability"); plt.title("Ground / optimal state probability"); plt.legend()

plt.subplot(2, 3, 3)
plt.plot(history[:, 5])
plt.xlabel("Iteration"); plt.ylabel("Approx ratio"); plt.title("Approximation ratio")

plt.subplot(2, 3, 4)
plt.plot(np.sqrt(np.maximum(history[:, 2], 0.0) / lamb))
plt.axhline(y=eps, color="r", linestyle="--", label="eps")
plt.xlabel("Iteration"); plt.ylabel("|P^T x - 1|"); plt.title("Budget violation"); plt.legend()

plt.subplot(2, 3, 5)
plt.plot(history[:, 1])
plt.xlabel("Iteration"); plt.ylabel("E_eval"); plt.title("Evaluation energy (no penalty)")

plt.subplot(2, 3, 6)
plt.plot(history[:, 6], label="gamma_0")
plt.plot(history[:, 7], label="beta_0")
plt.xlabel("Iteration"); plt.title("Parameters"); plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
colors = np.where(np.isin(np.arange(dim), idx_feasible), "tab:blue", "tab:red")
plt.bar(np.arange(dim), prob, color=colors)
plt.scatter(idx_ground, prob[idx_ground], color="green", zorder=3, label="ground")
plt.scatter([idx_optimal], [prob[idx_optimal]], color="orange", marker="x", zorder=3, label="optimal")
plt.axhline(y=1 / dim, color="r", linestyle="--")
plt.title("Final probability (blue = feasible)"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(sorted(prob), marker="o")
plt.axhline(y=1 / dim, color="r", linestyle="--")
plt.title("Sorted probabilities")
plt.show()

In [ ]:
assert False

# Method Comparison

In [ ]:
compare_configs = [("gradient", None), ("mcLachlan", "diagonalize"), ("mcLachlan", "tikhonov")]
runs = {}
for method, inv in compare_configs:
    OPTIMIZE_METHOD = method
    if inv is not None:
        INVERSE_METHOD = inv
    name = method if inv is None else f"{method}_{inv}"
    st = time.perf_counter()
    pars_c, hist_c, itt_c, ni_c = (run_mcLachlan if method == "mcLachlan" else run_gradient)(points_init)
    runs[name] = (hist_c, time.perf_counter() - st, ni_c)

rows = []
for name, (hist_c, tt, ni_c) in runs.items():
    rows.append([name, hist_c[-1, 0] * hamiltonian_boost, hist_c[-1, 3], hist_c[-1, 5], ni_c, tt])
print(pd.DataFrame(rows, columns=["Run", "Energy", "Prob_Ground", "Approx_ratio", "epochs", "time_s"]))

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
for name, (hist_c, _, _) in runs.items():
    plt.plot(hist_c[:, 0] * hamiltonian_boost, label=name)
plt.axhline(y=E0, color="r", linestyle="--", label="E0")
plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.title("Energy"); plt.legend()

plt.subplot(1, 2, 2)
for name, (hist_c, _, _) in runs.items():
    plt.plot(hist_c[:, 3], label=name)
plt.xlabel("Iteration"); plt.ylabel("P_ground"); plt.title("Ground-state probability"); plt.legend()
plt.show()

# Gradient: FD vs Parameter-Shift

In [ ]:
psr_configs = [("gradient", "fd"), ("gradient", "psr"), ("mcLachlan", "fd"), ("mcLachlan", "psr")]
runs_psr = {}
for method, gm in psr_configs:
    GRAD_METHOD = gm
    name = f"{method}_{gm}"
    st = time.perf_counter()
    pars_c, hist_c, itt_c, ni_c = (run_mcLachlan if method == "mcLachlan" else run_gradient)(points_init)
    runs_psr[name] = (hist_c, time.perf_counter() - st, ni_c, itt_c)
GRAD_METHOD = "fd"

rows = []
for name, (hist_c, tt, ni_c, itt_c) in runs_psr.items():
    rows.append([name, hist_c[-1, 0] * hamiltonian_boost, hist_c[-1, 3], hist_c[-1, 5], ni_c, itt_c.mean() * 1e3, tt])
print(pd.DataFrame(rows, columns=["Run", "Energy", "Prob_Ground", "Approx_ratio", "epochs", "iter_ms", "time_s"]))

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
for name, (hist_c, _, _, _) in runs_psr.items():
    plt.plot(hist_c[:, 0] * hamiltonian_boost, label=name)
plt.axhline(y=E0, color="r", linestyle="--", label="E0")
plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.title("Energy"); plt.legend()

plt.subplot(1, 2, 2)
for name, (hist_c, _, _, _) in runs_psr.items():
    plt.plot(hist_c[:, 3], label=name)
plt.xlabel("Iteration"); plt.ylabel("P_ground"); plt.title("Ground-state probability"); plt.legend()
plt.show()

# Estimation Choices (M / C routes)

Routes of `VarQITE_Estimation_Choices.md` that apply to this ansatz (parameters shared by a whole layer, so the $\pi/2$ shift rule M2 does not apply; the Hadamard-test route M3 costs $\sim T^2$ circuits per entry and is not run):

| | route | `A_METHOD` / `FID_STENCIL` | circuits per step ($p=2L$) | bias |
|---|---|---|---|---|
| M6 | statevector FD (reference, not measurable) | `dpsi` | $2p$ `get_state` | $O(\delta^2)$ |
| M1 | overlap 4-point stencil, $\delta_k=\kappa/\sqrt{A_{kk}}$, $A_{kk}=\mathrm{Var}(G_k)$ | `fidelity_kappa` / `central` | $2p(p-1) + 2p$ | $O(\kappa^2)$ |
| M1′ | overlap forward stencil, same shifts | `fidelity_kappa` / `forward` | $p(p-1)/2 + p + 2p$ | $O(\kappa)$ |
| M1u | overlap stencil, uniform `FID_SHIFT` | `fidelity` / `central` or `forward` | $2p^2$ or $p(p+1)/2$ | $O(\delta^2)$ / $O(\delta)$ |
| M4 | diagonal only | `diag` | $2p$ | drops all couplings |
| C1 / C2 | forward / central energy difference | `FD_SCHEME` | $p+1$ / $2p$ | $O(\delta)$ / $O(\delta^2)$ |
| C3 | term-wise shift rule | `GRAD_METHOD="psr"` | $2L(n+T)$ | none |

The cells below (1) compare the routes at the initial point, (2) run the flow with each and (3) emulate shot noise with `SHOTS`.


In [ ]:
est_configs = [("dpsi", "central", "central"), ("fidelity_kappa", "central", "forward"), ("fidelity_kappa", "forward", "forward"), ("fidelity", "central", "forward"), ("fidelity", "forward", "forward"), ("diag", "central", "forward")]
_save = (A_METHOD, FID_STENCIL, FD_SCHEME)
runs_est = {}
for A_METHOD, FID_STENCIL, FD_SCHEME in est_configs:
    name = f"{A_METHOD}/{FID_STENCIL[:3]} + C-{FD_SCHEME[:3]}"
    st = time.perf_counter()
    pars_c, hist_c, itt_c, ni_c = run_mcLachlan(points_init, pbar=False)
    runs_est[name] = (hist_c, time.perf_counter() - st, ni_c, itt_c, n_circuits_A() + (parameter_count + 1 if FD_SCHEME == "forward" else 2 * parameter_count), np.abs(pars_c).max())
    print(f"{name:34s} done: {ni_c} epochs, E {hist_c[-1, 0] * hamiltonian_boost:.6f}")
A_METHOD, FID_STENCIL, FD_SCHEME = _save

rows = [[name, h[-1, 0] * hamiltonian_boost, h[-1, 3], h[-1, 5], ni, nc, itt.mean() * 1e3, tt, int((h[:, 8] < 0).sum()), pm] for name, (h, tt, ni, itt, nc, pm) in runs_est.items()]
print(pd.DataFrame(rows, columns=["Run", "Energy", "Prob_Ground", "Approx_ratio", "epochs", "circuits/step", "iter_ms", "time_s", "R_min<0", "|params|max"]))

plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
for name, (h, *_) in runs_est.items():
    plt.plot(h[:, 0] * hamiltonian_boost, label=name)
plt.axhline(y=E0, color="r", linestyle="--", label="E0")
plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.title(f"Energy ({INVERSE_METHOD})"); plt.legend()
plt.subplot(1, 2, 2)
for name, (h, *_) in runs_est.items():
    plt.plot(h[:, 3], label=name)
plt.xlabel("Iteration"); plt.ylabel("P_ground"); plt.title("Ground-state probability"); plt.legend()
plt.show()


## Shot-noise emulation

`SHOTS = S` makes `fidelity` binomial, `var_diag` use $S$ sampled bitstrings and every energy in `build_C` Gaussian with $\sigma^2 = V_\tau/S$; noisy $A$ is projected to PSD before the ridge solve.
The plan's noise floor on each entry is $\approx 3.5\,\bar A/\sqrt S$ ($\kappa = 0.1$); entries below it are not resolved and must be regularized away, so `TIKHONOV_LAMBDA` should be compared with that floor.


In [ ]:
_save = (SHOTS, A_METHOD, FID_STENCIL, KAPPA, CAP_ANGLE, TIKHONOV_LAMBDA)
A_ref = build_A_dpsi(psi0, dps0); C_ref = -np.real(dps0.conj() @ (diag_H * psi0)); V0 = vtau(psi0)
sq = np.sqrt(np.outer(np.diag(A_ref), np.diag(A_ref)))
td_ref = solve_theta_dot(A_ref, C_ref)
n_draws = 10
print(f"{'S':>7s} {'A route':30s} | {'rel Frobenius':>13s} {'norm max':>9s} {'cond':>8s} {'floor 3.5A/sqrtS':>16s} | {'R_min<0':>7s} {'td rel err':>10s}")
for SHOTS in [1000, 10000, 100000]:
    for A_METHOD, FID_STENCIL, KAPPA, CAP_ANGLE in [("fidelity_kappa", "central", 0.1, 0.5), ("fidelity_kappa", "central", 0.01, 0.02), ("fidelity_kappa", "forward", 0.1, 0.5), ("diag", "central", 0.1, 0.5)]:
        errs, nmax, conds, rneg, tderr = [], [], [], 0, []
        for _ in range(n_draws):
            A_n = build_A(points_init); C_n = build_C(points_init); td = solve_theta_dot(A_n, C_n)
            errs.append(np.linalg.norm(A_n - A_ref) / np.linalg.norm(A_ref)); nmax.append((np.abs(A_n - A_ref) / sq).max()); conds.append(np.linalg.cond(A_n))
            rneg += (V0 - C_n @ td) < 0; tderr.append(np.abs(td - td_ref).max() / np.abs(td_ref).max())
        tag = A_METHOD + (f" {FID_STENCIL[:3]} k={KAPPA} cap={CAP_ANGLE}" if A_METHOD != "diag" else "")
        print(f"{SHOTS:7d} {tag:30s} | {np.mean(errs):13.1e} {np.mean(nmax):9.1e} {np.median(conds):8.1e} {3.5 * np.mean(np.diag(A_ref)) / np.sqrt(SHOTS):16.1e} | {rneg / n_draws:7.2f} {np.mean(tderr):10.1e}")

SHOTS, A_METHOD, FID_STENCIL, KAPPA, CAP_ANGLE = 10000, "fidelity_kappa", "central", 0.1, 0.5
runs_shots = {}
for TIKHONOV_LAMBDA in [1e-6, 1e-4, 1e-3]:
    st = time.perf_counter()
    pars_c, hist_c, itt_c, ni_c = run_mcLachlan(points_init, pbar=False)
    runs_shots[f"S={SHOTS} lambda={TIKHONOV_LAMBDA:.0e}"] = (hist_c, time.perf_counter() - st, ni_c, itt_c, np.abs(pars_c).max())
SHOTS, A_METHOD, FID_STENCIL, KAPPA, CAP_ANGLE, TIKHONOV_LAMBDA = _save
rows = [[name, h[-1, 0] * hamiltonian_boost, h[-1, 3], h[-1, 5], ni, itt.mean() * 1e3, tt, int((h[:, 8] < 0).sum()), pm] for name, (h, tt, ni, itt, pm) in runs_shots.items()]
print(pd.DataFrame(rows, columns=["Run", "Energy", "Prob_Ground", "Approx_ratio", "epochs", "iter_ms", "time_s", "R_min<0", "|params|max"]))


# Route Sweep (persisted)

The four routes are compared on the 7-asset / 14-qubit instance (`N_ASSETS=7`, `TARGET_QUBIT_IN=2`, p=5, Ramp init) over `lamb ∈ {0.0005, 0.005, 0.05}` and experiments 0–9 by the driver `varqite_routes.py` (this notebook's pipeline as a module) — launched with `./run_routes.sh`, a sequential queue (one simulator process at a time: concurrent 14-qubit runs starve each other on the GPU).

| tag | route | `OPTIMIZE_METHOD` / `A_METHOD` / `FID_STENCIL` / `FD_SCHEME` | circuits per step (p=10) |
|---|---|---|---|
| M6C2 | statevector FD reference + C2 central | mcLachlan / dpsi / – / central | 40 |
| GRAD | Adam + forward-difference gradient (old method) | gradient | 11 |
| M1fC1 | overlap forward stencil, κ=0.01 / cap 0.02, Var(G_k) diagonal + C1 forward | mcLachlan / fidelity_kappa / forward / forward | 86 |
| M4C1 | diagonal Var(G_k) only + C1 forward | mcLachlan / diag / – / forward | 31 |
| RAMP | linear ramp, **no optimization** (LR-QAOA / discrete annealing): paper schedule Δβ = 1.5, Δγ = 3.0 in circuit units (`PO_new_ApproxRatio.py` mode Ramp), β negated | ramp / – / – / – | 1 circuit in total, per p |
| RAMPB | linear ramp, no optimization: Δβ = 0.2, Δγ = 3.0 in **boosted** cost units (γ_i = Δγ·boost·(i+1)/p), tuned on exp 0 at p = 200 (`experiment/ramp_tune_20260916.py`), β negated | ramp / – / – / – | 1 circuit in total, per p |

Every run is stored in `experiment/exp_Q2_L{lamb}_q1.5/{report,expectation}_{tag}_Ramp_boost_Jh.{csv,npz}` (torch.ipynb format, key `A7_p5_E{e}_S0`; the report rows carry the route settings as extra columns). `plot_routes.py` builds the summary table and the figures shown below; cached runs are reused, `OVERWRITE=1 ./run_routes.sh` forces a rerun.

14-qubit re-check (`experiment/tune/`, exp 0, lamb 0.005, boost 72.27): the A spectrum spans 7e-11 … 6.5e-2 (γ-block 1e-3, β-block 5e-2 — much less anisotropic than the 6-qubit instance, whose coefficients are 50× smaller); κ=0.01 / cap 0.02 give a 2% normalized error on A with the forward stencil (4-point: 2e-4); C forward 2e-4 relative. `TIKHONOV_LAMBDA=1e-8` inverts FD noise (reference gap 0.16 vs 0.08), 1e-7…1e-5 are equivalent within instance noise (reference best at 1e-6, M1′ best at 1e-5, M4 insensitive) — the sweep uses 1e-6 for all routes. The gradient route reproduces torch.ipynb's stored run (E = +1.9746) exactly.

Cost note: at 14 qubits `observe(P_0)` expands $P_0=\prod_j\tfrac12(I+Z_j)$ into $2^{14}$ Pauli terms (≈120 ms per overlap, 5.8 s per M1′ step), so the sweep was run with the driver's simulator-only shortcut `EXACT_MEASURE="prob"`, which reads the same $|0\ldots0\rangle$ probability — and, for Var$(G_k)$, the measured-basis distribution of the truncated circuit — directly off the simulated circuit, i.e. the $S\to\infty$ limit of `sample` on that circuit (agrees with `observe` to 1e-15 / 1e-12; ≈1.06 s per M1′ step). The driver default is now `EXACT_MEASURE="observe"` (only `cudaq.observe` feeds the update, as required for a device port; `sample` + bitstring post-processing on hardware), and numpy's BLAS is pinned to one thread (16k-element dots otherwise wake 24 spinning threads).

**Driver trimmed (2026-09-16, after the sweep):** `varqite_routes.py` now implements only the device-executable routes — M4C1 / M4C2 (diagonal metric $\mathrm{Var}(G_k)$ from one sampled circuit per parameter, energy FD for $C$) and the GRAD baseline — with `SHOTS` runs using `cudaq.sample` + classical bitstring energies end to end, and no `get_state` in the update. The four-route driver that produced the persisted M6C2 / M1fC1 runs above is kept as `experiment/driver_4routes_20260916.py`.


**Linear-ramp baselines (2026-09-16).** `RAMP`/`RAMPB` evaluate the linear-ramp schedule $\gamma_i = \Delta\gamma\,(i+1)/p$, $\beta_i = -\Delta\beta\,(1-i/p)$ once — no parameter update, one device circuit — and are scanned over the layer count $p \in \{5, 10, 20, 50, 100, 200, 300, 500, 1000\}$ (`RAMP_LAYERS` in `run_routes.sh`; every $p$ is a row of the same report file, key `A7_p{p}_E{e}_S0`). Two facts decide how the schedule has to be read:

1. **Sign.** `kernel_qaoa_X` applies `rx(2β)` $= e^{-i\beta X}$ to $|+\rangle^{\otimes n}$, which is the *top* eigenstate of $+\sum_j X_j$. A ramp with positive angles therefore follows the top eigenstate adiabatically and anneals to the **maximum** of $H$: with the positive paper schedule the gap grows with $p$ and reaches $\max H - E_0 = 16.3$ at $p = 100$. Negating $\beta$ (mixer $-\sum X$, ground state $|+\rangle$) anneals to the ground state; negating $\gamma$ instead gives the complex-conjugate state (same probabilities, checked to $10^{-16}$). The ramp routes use `RAMP_SIGN = -1`; the optimizer routes keep the positive Ramp *init* (a starting point only — GRAD must keep reproducing the stored torch.ipynb run).
2. **Scale.** The circuit carries the unboosted coefficients ($\max|c| = 0.014$ at $\lambda = 0.005$, $0.0018$ at $0.0005$, $0.135$ at $0.05$), so $\Delta\gamma = 3$ rotates the cost by at most $0.08$ rad per layer while $\Delta\beta = 1.5$ is a 3 rad mixer kick: the paper schedule is mixer-dominated and stays above the $|+\rangle$ energy at $p = 5$ (gap 2.9 vs 3.4). In *boosted* units ($\gamma \to \gamma \cdot$ boost, i.e. $\Delta\gamma$ multiplies a cost Hamiltonian with $\max|c| = 1$) one schedule transfers across the three penalties: the exp-0 grid at $p = 200$ (`experiment/logs/ramp_tune_p200.log`) has its optimum at $(\Delta\beta, \Delta\gamma) = (0.3, 3)$, $(0.2, 4)$, $(0.2, 4)$ for $\lambda = 0.0005, 0.005, 0.05$, and $(0.2, 3.0)$ is within 10% of all three (gaps 0.011 / 0.018 / 0.013) — this is `RAMPB`. $\Delta\beta \geq 0.5$ or $\Delta\gamma \geq 6$ blow up at large $p$ (Trotter breakdown).

`plot_routes.py --route-layer RAMP=200 RAMPB=200 --ramp-scan` adds the ramp routes at $p = 200$ to the comparison figures (level lines: one value per instance) and writes `routes_ramp_scan_Q2_A7.{png,csv}`, the gap / $P_\mathrm{ground}$ / approximation ratio vs $p$ with M4C1 and GRAD ($p = 5$, their total circuit count in the legend) as reference lines.

**Ramp benchmark results (2026-09-16, `experiment/routes_ramp_scan_Q2_A7.csv`, 10 instances per penalty, mean gap $E-E_0$ in boosted units; optimizer routes at $p=5$).**

| λ | GRAD (≈3000 circuits) | M4C1 (2800–6000 circuits) | M6C2 (reference) | RAMP p=200 | RAMPB p=20 | RAMPB p=50 | RAMPB p=200 | RAMPB p=1000 |
|---|---|---|---|---|---|---|---|---|
| 0.0005 | 3.48 | 0.36 | 0.28 | 3.28 | 0.30 | 0.10 | **0.015** (P_gs 0.59) | 0.003 (P_gs 0.85) |
| 0.005 | 2.10 | 0.12 | 0.074 | 1.52 | 0.21 | 0.063 | **0.017** (P_gs 0.064) | 0.006 (P_gs 0.21) |
| 0.05 | 0.14 | 0.072 | 0.10 (median) | 0.13 | 0.19 | 0.047 | **0.012** (P_gs 0.015) | 0.006 (P_gs 0.029) |

- `RAMPB` (one circuit, no optimization) beats every optimized route at every penalty from $p = 50$ on, and at $p = 200$ its gap is 5–25× below M4C1's with a 10× smaller spread; the gain flattens beyond $p \approx 300$ for $\lambda \geq 0.005$ (0.012 → 0.006 from 300 to 1000) while $P_\mathrm{ground}$ keeps growing. At equal depth ($p = 5$) it is worse than M4C1 (gap ≈ 1.0–1.25): the schedule was tuned at $p = 200$ and small $p$ prefers a larger $\Delta\beta$ (exp-0 grid at $p = 5$: best gap 0.18 at $\Delta\beta = 0.5$, $\Delta\gamma = 1.4$ boosted units).
- `RAMP` (paper schedule in circuit units, sign-corrected) improves only slowly with depth (λ = 0.005: 3.0 → 0.70 from $p = 5$ to 1000) and stays above M4C1 except at λ = 0.05, where the boost is 7.4 and circuit units are closest to boosted units.
- At λ = 0.0005 the approximation ratio exceeds 1 for `RAMPB` because the ground state of $H$ slightly violates the ε-budget window (weak penalty): the ratio is measured against the feasible states, the gap against $E_0$.


In [ ]:
import importlib, plot_routes
importlib.reload(plot_routes)
ROUTE_LAMBS = [0.0005, 0.005, 0.05]
ROUTES_CMP = ["M6C2", "GRAD", "M1fC1", "M4C1", "RAMP", "RAMPB"]
RAMP_LAYER = 200                                     # ramp routes at this p in the comparison figures; the full p scan is in routes_ramp_scan
EXP = os.path.abspath("experiment")
df_routes, hist_routes = plot_routes.load(EXP, ROUTES_CMP, ROUTE_LAMBS, Q=2, q=1.5, init="Ramp", mode="Jh", suffix="", layer=5, n_assets=7, seed=0, route_layers={"RAMP": RAMP_LAYER, "RAMPB": RAMP_LAYER})
summary_routes = plot_routes.summary(df_routes)
print(summary_routes.to_string(index=False))
plot_routes.plot_traj(df_routes, hist_routes, ROUTES_CMP, ROUTE_LAMBS, "experiment/routes_traj_Q2_A7.png")
plot_routes.plot_final(df_routes, ROUTES_CMP, ROUTE_LAMBS, "experiment/routes_final_Q2_A7.png")
df_ramp, _ = plot_routes.load(EXP, ["RAMP", "RAMPB"], ROUTE_LAMBS, Q=2, q=1.5, init="Ramp", mode="Jh", suffix="", layer=5, n_assets=7, seed=0, all_layers=True)
ramp_scan = plot_routes.ramp_scan_table(df_ramp)
print(ramp_scan.to_string(index=False))
plot_routes.plot_ramp_scan(df_ramp, df_routes[df_routes["Route"].isin(["M4C1", "GRAD"])], ["RAMP", "RAMPB"], ["M4C1", "GRAD"], ROUTE_LAMBS, "experiment/routes_ramp_scan_Q2_A7.png")
from IPython.display import Image, display
for fig in ["routes_traj", "routes_final", "routes_ramp_scan"]:
    display(Image(f"experiment/{fig}_Q2_A7.png"))
